In [3]:
import torch
import torch.nn as nn
import math

### Data Pre-Processing

In [4]:
input_text="the cat jumped the dog jumped the cat jumped the dog"
words=input_text.split()
vocab=list(set(words))
word2idx={word:idx for idx,word in enumerate(vocab)}
idx2word={idx:word for word,idx in word2idx.items()}
input_token=torch.tensor([word2idx[i] for i in words[:-1]],dtype=torch.long)
output_token=torch.tensor([word2idx[i] for i in words[1:]],dtype=torch.long)
input_token=input_token.unsqueeze(0)
output_token=output_token.unsqueeze(0)

### Positional Encoding

In [5]:
class Encoding(nn.Module):
    def __init__(self,maximum_size,feature_size):
        super().__init__()
        encoding=torch.zeros(maximum_size,feature_size)
        print(encoding.shape)
        position=torch.arange(maximum_size).unsqueeze(1)
        div_term=torch.exp(torch.arange(0,feature_size,2)*(-math.log(10000.0)/feature_size))
        encoding[:,0::2]=torch.sin(position*div_term)
        encoding[:,1::2]=torch.cos(position*div_term)
        self.register_buffer("encoding",encoding)
    def forward(self,x):
        size=x.shape[1]
        return self.encoding[:size]+x

### Multi Head Attention

In [6]:
class Attention(nn.Module):
    def __init__(self,features,num_heads):
        super().__init__()
        self.head_features=features//num_heads
        self.num_heads=num_heads
        self.q=nn.Linear(features,features)
        self.k=nn.Linear(features,features)
        self.v=nn.Linear(features,features)
    def forward(self,x):
        q=self.q(x)
        k=self.k(x)
        v=self.v(x)
        q=q.view(q.size(0),q.size(1),self.num_heads,self.head_features).transpose(1,2)
        k=k.view(k.size(0),k.size(1),self.num_heads,self.head_features).transpose(1,2)
        v=v.view(v.size(0),v.size(1),self.num_heads,self.head_features).transpose(1,2)
        score=q@k.transpose(-2,-1)/self.head_features**0.5
        mask=torch.tril(torch.ones(x.shape[1],x.shape[1])).to(x.device)
        score=score.masked_fill(mask==0,float("-inf"))
        scaled_score=score.softmax(dim=-1)
        attention=scaled_score@v
        attention=attention.transpose(1,2)
        attention=attention.contiguous().view(attention.size(0),attention.size(1),-1)
        return attention

### Resnet Feed Forward Network

In [7]:
class FNN(nn.Module):
    def __init__(self,features,num_heads):
        super().__init__()
        self.features=features
        self.attention=Attention(self.features,num_heads)
        self.norm1=nn.LayerNorm(features)
        self.norm2=nn.LayerNorm(features)
        self.model=nn.Sequential(
            nn.Linear(features,features*4),
            nn.GELU(),
            nn.Linear(features*4,features)
        )
    def forward(self,x):
        attention=self.attention(x)
        x = self.norm1(x + attention)
        ff = self.model(x)
        x = self.norm2(x + ff)
        return x

In [8]:
features=16
num_heads=4

encoder=nn.Embedding(len(vocab),features)
positional_encoding=Encoding(100,features)

encoded_input=positional_encoding(encoder(input_token))
encoded_output=output_token
model=FNN(features,num_heads)

output=model(encoded_input)
output_layer=nn.Linear(features,len(vocab))
output=output_layer(output)
predicted_idx=torch.argmax(output,dim=-1)
print(predicted_idx.shape  )
print(output.shape)

predicted_words=[idx2word[idx.item()]for idx in predicted_idx[0]]
target_words=[idx2word[idx.item()]for idx in output_token[0]]

torch.Size([100, 16])
torch.Size([1, 10])
torch.Size([1, 10, 4])


### Training Model

In [9]:
loss_fn=nn.CrossEntropyLoss()
loss=loss_fn(output.view(-1,len(vocab)),output_token.view(-1))
optimizer=torch.optim.Adam(list(encoder.parameters())+list(positional_encoding.parameters())+list(model.parameters())+list(output_layer.parameters()),lr=0.001)
for i in range(1000):
    optimizer.zero_grad()
    embedded=encoder(input_token)
    embedded=positional_encoding(embedded)
    hidden=model(embedded)
    final=output_layer(hidden)
    loss=loss_fn(final.view(-1, len(vocab)),output_token.view(-1))
    loss.backward()
    optimizer.step()
    if i%100==0:print(i,":", loss.item())

0 : 1.3367730379104614
100 : 0.08840818703174591
200 : 0.029859129339456558
300 : 0.015174278989434242
400 : 0.009148674085736275
500 : 0.006097675301134586
600 : 0.0043421476148068905
700 : 0.0032404891680926085
800 : 0.0025042498018592596
900 : 0.0019882579799741507


### Evaluating Trained Model

In [10]:
with torch.no_grad():
    embedded=encoder(input_token)
    embedded=positional_encoding(embedded)
    hidden=model(embedded)
    final=output_layer(hidden)
    predicted_idx=torch.argmax(final,dim=-1)
    predicted_words=[idx2word[idx.item()]for idx in predicted_idx[0]]
target_words = [idx2word[idx.item()]for idx in output_token[0]]
print("Input:", input_text)
print("Predicted:", predicted_words)
print("Target:", target_words)
accuracy=(predicted_idx==output_token).float().mean()
print("Accuracy:",accuracy.item())

Input: the cat jumped the dog jumped the cat jumped the dog
Predicted: ['cat', 'jumped', 'the', 'dog', 'jumped', 'the', 'cat', 'jumped', 'the', 'dog']
Target: ['cat', 'jumped', 'the', 'dog', 'jumped', 'the', 'cat', 'jumped', 'the', 'dog']
Accuracy: 1.0


In [ ]:
prompt="the"
prompt=prompt.split()
input_tensor=torch.tensor([word2idx[i] for i in prompt]).unsqueeze(0)
with torch.no_grad():
    for iter in range(12):
        embedded=encoder(input_tensor)
        hidden=model(embedded)
        final=output_layer(hidden)
        next_token=torch.argmax(final[:,-1,:],dim=-1)
        input_tensor=torch.cat([input_tensor,next_token.unsqueeze(0)],dim=1)
print("Predicted:", [idx2word[idx.item()] for idx in input_tensor[0]])

Predicted: ['the', 'cat', 'jumped', 'the', 'dog', 'jumped', 'the', 'dog', 'jumped']
